In [56]:
import pandas as pd
import plotly.express as px

# KCGS 데이터 불러오기
df_kcgs = pd.read_csv("../data/kcgs.csv")

df_kcgs.shape

(12081, 13)

# KCGS ESG 평가등급 분석

KCGS 데이터를 활용해 국내 기업의 ESG 평가등급이 연도별로 어떻게 변화했는지 확인하고, 이후 환경등급을 중심으로 실제 환경성과와의 관계를 분석하려고 한다.

In [57]:
# 컬럼 확인
print(df_kcgs.columns)

# 평가연도 범위 확인
print('최초 평가연도:', df_kcgs['평가년도'].min())
print('최근 평가연도:', df_kcgs['평가년도'].max())

# 연도별 데이터 수 확인
df_kcgs['평가년도'].value_counts().sort_index()

Index(['NO', '기업명', '기업코드', 'ESG등급', '환경', '사회', '지배구조', '평가년도', 'ESG 등급조정',
       '환경 등급조정', '사회 등급조정', '지배구조 등급조정', 'Unnamed: 12'],
      dtype='str')
최초 평가연도: 2012.0
최근 평가연도: 2025.0


평가년도
2012.0     706
2013.0     690
2014.0     692
2015.0     693
2016.0     710
2017.0     732
2018.0     726
2019.0     924
2020.0     963
2021.0    1001
2022.0    1033
2023.0    1049
2024.0    1066
2025.0    1091
Name: count, dtype: int64

In [58]:
# ESG와 각 영역의 등급 종류 확인
print('ESG 등급')
print(df_kcgs['ESG등급'].value_counts(dropna=False))

print('\n환경 등급')
print(df_kcgs['환경'].value_counts(dropna=False))

print('\n사회 등급')
print(df_kcgs['사회'].value_counts(dropna=False))

print('\n지배구조 등급')
print(df_kcgs['지배구조'].value_counts(dropna=False))

ESG 등급
ESG등급
미공개     4375
B+      1719
C       1661
A       1295
B       1248
D       1234
등급없음     411
A+       133
NaN        5
Name: count, dtype: int64

환경 등급
환경
미공개     4117
B+      1860
D       1805
C       1448
A       1230
B       1009
등급없음     411
A+       196
NaN        5
Name: count, dtype: int64

사회 등급
사회
미공개     4075
B+      1666
A       1341
C       1308
D       1179
B       1095
A+      1001
등급없음     411
NaN        5
Name: count, dtype: int64

지배구조 등급
지배구조
B       3514
B+      2662
C       2611
D       1338
A       1186
미공개      593
A+       170
NaN        5
S          1
등급없음       1
Name: count, dtype: int64


In [59]:
# 평가연도별 ESG 등급 분포
esg_year_table = pd.crosstab(
    df_kcgs['평가년도'],
    df_kcgs['ESG등급']
)

esg_year_table

ESG등급,A,A+,B,B+,C,D,등급없음,미공개
평가년도,,,,,,,,
2012.0,37,6,0,81,0,0,0,582
2013.0,33,6,0,90,0,0,0,561
2014.0,43,2,0,88,0,0,0,559
2015.0,27,1,0,77,0,0,0,588
2016.0,33,8,0,106,0,0,0,563
2017.0,38,5,0,115,0,0,0,574
2018.0,41,8,0,100,0,0,0,577
2019.0,50,7,308,139,0,0,49,371
2020.0,91,14,318,147,309,29,55,0


In [60]:
# 평가연도별 환경 등급 분포
env_year_table = pd.crosstab(
    df_kcgs['평가년도'],
    df_kcgs['환경']
)

env_year_table

환경,A,A+,B,B+,C,D,등급없음,미공개
평가년도,,,,,,,,
2012.0,30,10,0,131,0,0,0,535
2013.0,36,6,0,121,0,0,0,527
2014.0,40,6,0,135,0,0,0,511
2015.0,40,5,0,141,0,0,0,507
2016.0,62,3,0,132,0,0,0,513
2017.0,58,4,0,144,0,0,0,526
2018.0,52,7,0,144,0,0,0,523
2019.0,36,5,265,94,0,0,49,475
2020.0,58,10,243,103,198,296,55,0


### 데이터 확인

환경등급의 연도별 분포를 확인한 결과, 2012~2018년에는 `미공개` 비중이 높고 일부 등급만 확인되었다. 2019년에도 미공개 기업이 많이 남아 있었지만, 2020년부터는 `A+`부터 `D`까지 전체 등급이 나타나고 `미공개` 값도 확인되지 않았다.

따라서 서로 다른 공개 구조로 인한 왜곡을 줄이기 위해, 환경등급의 연도별 변화는 2020~2025년을 중심으로 분석하기로 했다.

In [61]:
# 환경등급 추세 분석에 사용할 2020~2025년 데이터
env_recent = df_kcgs[
    df_kcgs['평가년도'].between(2020, 2025)
].copy()

# 실제 등급만 사용
env_recent = env_recent[
    env_recent['환경'].isin(['A+', 'A', 'B+', 'B', 'C', 'D'])
]

# 연도별 환경등급 비율 계산
env_year_pct = pd.crosstab(
    env_recent['평가년도'],
    env_recent['환경'],
    normalize='index'
).mul(100).round(1)

# 등급 순서 정리
env_year_pct = env_year_pct[
    ['A+', 'A', 'B+', 'B', 'C', 'D']
]

env_year_pct

환경,A+,A,B+,B,C,D
평가년도,,,,,,
2020.0,1.1,6.4,11.3,26.8,21.8,32.6
2021.0,1.2,12.3,13.2,20.4,20.4,32.6
2022.0,0.9,9.0,11.0,9.5,21.7,47.8
2023.0,4.0,18.8,14.8,6.8,29.8,25.8
2024.0,4.0,20.8,15.9,6.5,28.6,24.3
2025.0,4.0,21.5,17.4,8.1,26.0,23.1


In [62]:
# 그래프용 데이터 형태로 변환
env_year_plot = (
    env_year_pct
    .reset_index()
    .melt(
        id_vars='평가년도',
        var_name='환경등급',
        value_name='비율'
    )
)

# 연도별 환경등급 분포 시각화
fig = px.bar(
    env_year_plot,
    x='평가년도',
    y='비율',
    color='환경등급',
    barmode='stack',
    title='2020~2025년 KCGS 환경등급 분포'
)

fig.update_layout(
    xaxis_title='평가연도',
    yaxis_title='비율(%)',
    yaxis_range=[0, 100]
)

fig.show()

In [63]:
# 기업코드와 평가년도가 중복되는 행이 있는지 확인
duplicate_count = df_kcgs.duplicated(
    subset=['기업코드', '평가년도']
).sum()

print('기업코드 + 평가년도 중복 행 수:', duplicate_count)

기업코드 + 평가년도 중복 행 수: 4


In [64]:
# 기업코드 결측치 확인
df_kcgs['기업코드'].isna().sum()

np.int64(5)

In [65]:
# 기업코드 + 평가년도 중복 행 확인
duplicate_rows = df_kcgs[
    df_kcgs.duplicated(
        subset=['기업코드', '평가년도'],
        keep=False
    )
].sort_values(['기업코드', '평가년도'])

duplicate_rows

,NO,기업명,기업코드,ESG등급,환경,사회,지배구조,평가년도,ESG 등급조정,환경 등급조정,사회 등급조정,지배구조 등급조정,Unnamed: 12
1517,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1819,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4633,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5837,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6455,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [66]:
  # 기업코드가 없는 행 확인
  missing_code_rows = df_kcgs[
      df_kcgs['기업코드'].isna()
  ]

  missing_code_rows

,NO,기업명,기업코드,ESG등급,환경,사회,지배구조,평가년도,ESG 등급조정,환경 등급조정,사회 등급조정,지배구조 등급조정,Unnamed: 12
1517,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1819,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4633,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5837,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6455,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [67]:
# 모든 값이 비어 있는 행 제거
df_kcgs = df_kcgs.dropna(how='all').copy()

# 불필요한 Unnamed 컬럼 제거
df_kcgs = df_kcgs.loc[
    :, ~df_kcgs.columns.str.startswith('Unnamed:')
]

# 데이터 크기 다시 확인
df_kcgs.shape

(12076, 12)

In [68]:
# 기업코드 + 평가년도 중복 재확인
print(
    '중복:',
    df_kcgs.duplicated(
        subset=['기업코드', '평가년도']
    ).sum()
)

# 기업코드 결측 재확인
print(
    '기업코드 결측:',
    df_kcgs['기업코드'].isna().sum()
)

중복: 0
기업코드 결측: 0


## 2021~2022년 동일 기업의 환경등급 변화

2022년에 D등급 비중이 크게 증가한 원인을 확인하기 위해, 2021년과 2022년에 모두 평가된 동일 기업의 환경등급 변화를 비교했다.

In [69]:
# 2021년 환경등급 데이터
env_2021 = df_kcgs[
    df_kcgs['평가년도'] == 2021
][['기업코드', '기업명', '환경']].copy()

# 2022년 환경등급 데이터
env_2022 = df_kcgs[
    df_kcgs['평가년도'] == 2022
][['기업코드', '기업명', '환경']].copy()

# 같은 기업코드를 기준으로 2021년과 2022년 데이터 연결
env_21_22 = env_2021.merge(
    env_2022,
    on='기업코드',
    how='inner',
    suffixes=('_2021', '_2022')
)

print('2021·2022년 모두 평가된 기업 수:', len(env_21_22))

2021·2022년 모두 평가된 기업 수: 964


In [70]:
# 분석에 사용할 실제 환경등급
valid_grades = ['A+', 'A', 'B+', 'B', 'C', 'D']

env_21_22_valid = env_21_22[
    env_21_22['환경_2021'].isin(valid_grades) &
    env_21_22['환경_2022'].isin(valid_grades)
].copy()

print('등급 비교 가능한 기업 수:', len(env_21_22_valid))

# 2021년 → 2022년 환경등급 이동 확인
grade_transition = pd.crosstab(
    env_21_22_valid['환경_2021'],
    env_21_22_valid['환경_2022']
)

# 등급 순서 맞추기
grade_transition = grade_transition.reindex(
    index=valid_grades,
    columns=valid_grades,
    fill_value=0
)

grade_transition

등급 비교 가능한 기업 수: 909


환경_2022,A+,A,B+,B,C,D
환경_2021,,,,,,
A+,2,8,1,0,0,0
A,7,63,39,4,1,0
B+,0,9,40,39,32,3
B,0,4,17,23,92,55
C,0,0,2,18,42,124
D,0,0,3,8,34,239


In [71]:
# 2021년, 2022년 원본 환경등급 개수 재확인
print('2021년 환경등급')
print(
    df_kcgs[df_kcgs['평가년도'] == 2021]['환경']
    .value_counts()
)

print('\n2022년 환경등급')
print(
    df_kcgs[df_kcgs['평가년도'] == 2022]['환경']
    .value_counts()
)

# 현재 만들어진 데이터의 연도 확인
print('\nenv_2021 행 수:', len(env_2021))
print('env_2022 행 수:', len(env_2022))

# 이동표의 행 합계 확인
print('\n이동표 2021년 등급별 합계')
print(grade_transition.sum(axis=1))

# 이동표의 열 합계 확인
print('\n이동표 2022년 등급별 합계')
print(grade_transition.sum(axis=0))

2021년 환경등급
환경
D       308
B       193
C       193
B+      125
A       116
등급없음     55
A+       11
Name: count, dtype: int64

2022년 환경등급
환경
D       466
C       211
B+      107
B        93
A        88
등급없음     59
A+        9
Name: count, dtype: int64

env_2021 행 수: 1001
env_2022 행 수: 1033

이동표 2021년 등급별 합계
환경_2021
A+     11
A     114
B+    123
B     191
C     186
D     284
dtype: int64

이동표 2022년 등급별 합계
환경_2022
A+      9
A      84
B+    102
B      92
C     201
D     421
dtype: int64


In [72]:
# 2022년에 D등급이 된 기업들의 2021년 등급 확인
d_2022_origin = (
    env_21_22_valid[
        env_21_22_valid['환경_2022'] == 'D'
    ]['환경_2021']
    .value_counts()
    .reindex(valid_grades, fill_value=0)
)

d_2022_origin

환경_2021
A+      0
A       0
B+      3
B      55
C     124
D     239
Name: count, dtype: int64

In [73]:
# 환경등급의 순서만 숫자로 지정
# 평균점수를 계산하려는 것이 아니라 상향/하향 여부만 판단하기 위한 값
grade_rank = {
    'D': 1,
    'C': 2,
    'B': 3,
    'B+': 4,
    'A': 5,
    'A+': 6
}

# 2021년과 2022년 등급 순서 변환
env_21_22_valid['rank_2021'] = (
    env_21_22_valid['환경_2021'].map(grade_rank)
)

env_21_22_valid['rank_2022'] = (
    env_21_22_valid['환경_2022'].map(grade_rank)
)

# 등급 변화 방향 구분
def grade_change(row):
    if row['rank_2022'] > row['rank_2021']:
        return '상향'
    elif row['rank_2022'] < row['rank_2021']:
        return '하향'
    else:
        return '유지'

env_21_22_valid['등급변화'] = (
    env_21_22_valid.apply(grade_change, axis=1)
)

# 기업 수와 비율 확인
change_count = env_21_22_valid['등급변화'].value_counts()

change_pct = (
    env_21_22_valid['등급변화']
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

pd.DataFrame({
    '기업수': change_count,
    '비율(%)': change_pct
})

,기업수,비율(%)
등급변화,,
유지,409,45.0
하향,398,43.8
상향,102,11.2


### 분석 결과

2021년과 2022년에 모두 환경등급을 받은 동일 기업 909개를 비교한 결과, 45.0%는 등급을 유지했고 43.8%는 등급이 하향되었다. 반면 등급이 상향된 기업은 11.2%에 그쳤다.

앞서 2022년에 D등급 비중이 크게 증가한 현상은 신규 평가기업의 유입뿐만 아니라, 기존 평가기업의 등급 하향에서도 나타난 것을 확인했다.

다만 이러한 등급 변화가 기업의 실제 환경성과 악화를 의미한다고 단정할 수는 없다. 평가기준이나 평가모형의 변화가 등급에 영향을 줬을 가능성도 있기 때문에, 이후 연도의 등급 이동도 함께 확인하기로 했다.

In [74]:
# 2022년 환경등급
env_2022 = df_kcgs[
    df_kcgs['평가년도'] == 2022
][['기업코드', '기업명', '환경']].copy()

# 2023년 환경등급
env_2023 = df_kcgs[
    df_kcgs['평가년도'] == 2023
][['기업코드', '기업명', '환경']].copy()

# 동일 기업 연결
env_22_23 = env_2022.merge(
    env_2023,
    on='기업코드',
    how='inner',
    suffixes=('_2022', '_2023')
)

# 실제 등급만 사용
env_22_23_valid = env_22_23[
    env_22_23['환경_2022'].isin(valid_grades) &
    env_22_23['환경_2023'].isin(valid_grades)
].copy()

# 등급 순서 변환
env_22_23_valid['rank_2022'] = env_22_23_valid['환경_2022'].map(grade_rank)
env_22_23_valid['rank_2023'] = env_22_23_valid['환경_2023'].map(grade_rank)

# 등급 변화 방향
env_22_23_valid['등급변화'] = env_22_23_valid.apply(
    lambda row: '상향'
    if row['rank_2023'] > row['rank_2022']
    else '하향'
    if row['rank_2023'] < row['rank_2022']
    else '유지',
    axis=1
)

change_22_23 = pd.DataFrame({
    '기업수': env_22_23_valid['등급변화'].value_counts(),
    '비율(%)': env_22_23_valid['등급변화']
        .value_counts(normalize=True)
        .mul(100)
        .round(1)
})

change_22_23

,기업수,비율(%)
등급변화,,
상향,497,52.8
유지,430,45.7
하향,14,1.5


### 분석 결과

2022년과 2023년에 모두 환경등급을 받은 동일 기업을 비교한 결과, 52.8%의 기업이 등급이 상향되었고 45.7%는 기존 등급을 유지했다. 반면 등급이 하향된 기업은 1.5%에 불과했다.

2021→2022년에는 하향 기업이 43.8%였던 것과 비교하면, 2022→2023년에는 등급 이동 방향이 크게 반전된 것을 확인할 수 있었다.

따라서 2023년 D등급 비중 감소는 단순히 평가대상 기업이 바뀐 영향뿐만 아니라, 기존 평가기업의 환경등급 상향에서도 나타났다.

다만 이러한 변화는 KCGS 평가등급의 변화이며, 기업의 실제 환경성과가 같은 수준으로 개선되었다는 의미로 해석하지는 않았다.

In [75]:
# 2022년 → 2023년 환경등급 이동표
transition_22_23 = pd.crosstab(
    env_22_23_valid['환경_2022'],
    env_22_23_valid['환경_2023']
)

transition_22_23 = transition_22_23.reindex(
    index=valid_grades,
    columns=valid_grades,
    fill_value=0
)

transition_22_23

환경_2023,A+,A,B+,B,C,D
환경_2022,,,,,,
A+,7,2,0,0,0,0
A,23,60,5,0,0,0
B+,9,75,23,0,0,0
B,0,28,47,14,4,0
C,0,14,60,34,97,3
D,0,2,7,18,180,229


### 분석 결과

2022년과 2023년의 환경등급 이동을 확인한 결과, 가장 큰 변화는 하위 등급에서 나타났다.

2022년 D등급 기업 중 229개는 D등급을 유지했지만, 180개가 C등급으로 상승했고 일부 기업은 B, B+, A등급까지 이동했다. 비교 가능한 D등급 기업 436개 중 약 47.5%가 2023년에 상향된 것으로 나타났다.

C등급에서도 비슷한 흐름이 나타났다. C등급을 유지한 기업은 97개였지만, B+로 60개, B로 34개, A로 14개가 상승했다.

따라서 2023년 환경등급 분포의 변화는 하위 등급 기업들이 상위 등급으로 이동한 영향이 크게 나타난 것으로 볼 수 있다.

다만 이는 KCGS 평가등급의 이동을 의미하며, 기업의 실제 환경성과가 같은 수준으로 개선되었다고 해석하지는 않았다.

## 2023~2024년 동일 기업의 환경등급 변화

2022~2023년에 나타난 큰 폭의 등급 상향이 이후에도 이어졌는지 확인하기 위해, 2023년과 2024년에 모두 평가된 동일 기업의 환경등급 변화를 비교했다.

In [76]:
# 2023년 환경등급 데이터
env_2023 = df_kcgs[
    df_kcgs['평가년도'] == 2023
][['기업코드', '기업명', '환경']].copy()

# 2024년 환경등급 데이터
env_2024 = df_kcgs[
    df_kcgs['평가년도'] == 2024
][['기업코드', '기업명', '환경']].copy()

# 동일 기업 연결
env_23_24 = env_2023.merge(
    env_2024,
    on='기업코드',
    how='inner',
    suffixes=('_2023', '_2024')
)

# 두 연도 모두 실제 등급이 있는 기업만 사용
env_23_24_valid = env_23_24[
    env_23_24['환경_2023'].isin(valid_grades) &
    env_23_24['환경_2024'].isin(valid_grades)
].copy()

print('등급 비교 가능한 기업 수:', len(env_23_24_valid))

등급 비교 가능한 기업 수: 956


In [77]:
# 등급 순서를 숫자로 변환
env_23_24_valid['rank_2023'] = (
    env_23_24_valid['환경_2023'].map(grade_rank)
)

env_23_24_valid['rank_2024'] = (
    env_23_24_valid['환경_2024'].map(grade_rank)
)

# 등급 변화 방향 구분
env_23_24_valid['등급변화'] = env_23_24_valid.apply(
    lambda row:
        '상향' if row['rank_2024'] > row['rank_2023']
        else '하향' if row['rank_2024'] < row['rank_2023']
        else '유지',
    axis=1
)

# 기업 수와 비율 확인
change_23_24 = pd.DataFrame({
    '기업수': env_23_24_valid['등급변화'].value_counts(),
    '비율(%)': (
        env_23_24_valid['등급변화']
        .value_counts(normalize=True)
        .mul(100)
        .round(1)
    )
})

change_23_24

,기업수,비율(%)
등급변화,,
유지,670,70.1
상향,184,19.2
하향,102,10.7


### 분석 결과

2023년과 2024년에 모두 평가된 동일 기업을 비교한 결과, 70.1%가 기존 환경등급을 유지했다.

등급이 상향된 기업은 19.2%, 하향된 기업은 10.7%로 나타났다.

2021~2022년에는 하향 비율이 높았고, 2022~2023년에는 상향 비율이 크게 증가했던 것과 달리, 2023~2024년에는 등급을 유지한 기업의 비중이 가장 높았다.

따라서 2022~2023년에 나타났던 큰 폭의 등급 이동은 이후에는 다소 완화되고, 환경등급 분포가 비교적 안정된 모습을 보였다.

In [78]:
# 2024년 환경등급 데이터
env_2024 = df_kcgs[
    df_kcgs['평가년도'] == 2024
][['기업코드', '기업명', '환경']].copy()

# 2025년 환경등급 데이터
env_2025 = df_kcgs[
    df_kcgs['평가년도'] == 2025
][['기업코드', '기업명', '환경']].copy()

# 동일 기업 연결
env_24_25 = env_2024.merge(
    env_2025,
    on='기업코드',
    how='inner',
    suffixes=('_2024', '_2025')
)

# 실제 환경등급만 사용
env_24_25_valid = env_24_25[
    env_24_25['환경_2024'].isin(valid_grades) &
    env_24_25['환경_2025'].isin(valid_grades)
].copy()

# 등급 순서를 숫자로 변환
env_24_25_valid['rank_2024'] = (
    env_24_25_valid['환경_2024'].map(grade_rank)
)

env_24_25_valid['rank_2025'] = (
    env_24_25_valid['환경_2025'].map(grade_rank)
)

# 상향 / 유지 / 하향 구분
env_24_25_valid['등급변화'] = env_24_25_valid.apply(
    lambda row:
        '상향' if row['rank_2025'] > row['rank_2024']
        else '하향' if row['rank_2025'] < row['rank_2024']
        else '유지',
    axis=1
)

change_24_25 = pd.DataFrame({
    '기업수': env_24_25_valid['등급변화'].value_counts(),
    '비율(%)': (
        env_24_25_valid['등급변화']
        .value_counts(normalize=True)
        .mul(100)
        .round(1)
    )
})

change_24_25

,기업수,비율(%)
등급변화,,
유지,733,75.6
상향,154,15.9
하향,83,8.6


### 분석 결과

2024년과 2025년에 모두 평가된 동일 기업을 비교한 결과, 75.6%가 기존 환경등급을 유지했다.

등급이 상향된 기업은 15.9%, 하향된 기업은 8.6%로 나타났다.

2023~2024년에 이어 2024~2025년에도 등급 유지 비율이 가장 높게 나타나면서, 2022~2023년에 보였던 큰 폭의 등급 이동은 이후 점차 줄어든 것을 확인할 수 있었다.

In [79]:
# 각 기간의 상향·유지·하향 비율을 계산하는 함수
def get_change_pct(df):
    return (
        df['등급변화']
        .value_counts(normalize=True)
        .mul(100)
        .reindex(['상향', '유지', '하향'], fill_value=0)
        .round(1)
    )

# 기간별 등급 변화 비율 정리
change_summary = pd.DataFrame({
    '2021→2022': get_change_pct(env_21_22_valid),
    '2022→2023': get_change_pct(env_22_23_valid),
    '2023→2024': get_change_pct(env_23_24_valid),
    '2024→2025': get_change_pct(env_24_25_valid)
}).T

change_summary

등급변화,상향,유지,하향
2021→2022,11.2,45.0,43.8
2022→2023,52.8,45.7,1.5
2023→2024,19.2,70.1,10.7
2024→2025,15.9,75.6,8.6


In [80]:
# 그래프용 형태로 변환
change_plot = (
    change_summary
    .reset_index()
    .rename(columns={'index': '기간'})
    .melt(
        id_vars='기간',
        var_name='등급변화',
        value_name='비율'
    )
)

fig = px.bar(
    change_plot,
    x='기간',
    y='비율',
    color='등급변화',
    barmode='group',
    text='비율',
    title='연도별 동일 기업의 KCGS 환경등급 변화'
)

fig.update_traces(
    texttemplate='%{text}%',
    textposition='outside'
)

fig.update_layout(
    xaxis_title='비교 기간',
    yaxis_title='비율(%)'
)

fig.show()

### 분석 결과

동일 기업의 환경등급 변화를 연도별로 비교한 결과, 2021~2023년과 이후 기간에서 다른 흐름이 나타났다.

2021→2022년에는 하향 기업이 43.8%로 높았고, 상향 기업은 11.2%에 그쳤다. 반대로 2022→2023년에는 상향 기업이 52.8%로 크게 증가했고 하향 기업은 1.5%로 감소했다.

하지만 2023년 이후에는 큰 폭의 이동보다는 기존 등급을 유지하는 기업이 많아졌다. 유지 비율은 2023→2024년 70.1%, 2024→2025년 75.6%로 증가했다.

따라서 KCGS 환경등급은 2022~2023년에 큰 변동이 나타났지만, 이후에는 등급 이동 폭이 줄어들면서 비교적 안정적인 분포를 보였다.

다만 각 기간마다 비교 가능한 기업의 구성이 조금씩 다르고, 환경등급의 변화가 기업의 실제 환경성과 변화와 동일한 의미는 아니기 때문에 등급 변화를 실제 환경성과 개선이나 악화로 직접 해석하지는 않았다.